In [ ]:
from pathlib import Path
import runpy

def _find_notebook_bootstrap(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "notebooks" / "_bootstrap.py"
        if direct.exists():
            return direct
        nested = candidate / "abstractgraph-ml" / "notebooks" / "_bootstrap.py"
        if nested.exists():
            return nested
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_find_notebook_bootstrap(Path.cwd())))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Estimator wrapper demo

Quick sanity check for the `Estimator` that combines `AbstractGraphTransformer`, an optional manifold step, and a downstream scikit estimator.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from abstractgraph.utils import plot_dataset_method_bars, plot_pareto
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", message="n_jobs value .*overridden")
os.environ.setdefault("KMP_WARNINGS", "0")
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split

from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph_ml.estimators import GraphEstimator


def compute_metrics(y_true, y_pred, proba=None):
    acc = accuracy_score(y_true, y_pred)
    auc = None
    ap = None
    if proba is not None:
        classes = np.unique(y_true)
        if len(classes) > 1:
            # Handle binary separately for cleaner metrics.
            try:
                if len(classes) == 2:
                    pos_idx = 1 if proba.shape[1] > 1 else 0
                    scores = proba[:, pos_idx]
                    auc = roc_auc_score(y_true, scores)
                    ap = average_precision_score(y_true, scores)
                else:
                    auc = roc_auc_score(y_true, proba, multi_class="ovr")
                    y_bin = label_binarize(y_true, classes=classes)
                    ap = average_precision_score(y_bin, proba, average="macro")
            except Exception:
                auc = auc if "auc" in locals() else None
                ap = ap if "ap" in locals() else None
    return acc, auc, ap

def plot_decomposition_comparison(report_df, metric="accuracy", ax=None, auto_zero=True):
    """Grouped bar plot to compare decompositions side by side per assay."""
    if report_df is None or report_df.empty:
        raise ValueError("report_df is empty")
    if metric not in report_df.columns:
        raise ValueError(f"metric {metric} not in report_df")
    pivot = report_df.pivot_table(index="assay_id", columns="decomposition", values=metric, aggfunc="mean")
    if ax is None:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(10, 4))
        show = True
    else:
        show = False
    color_cycle = plt.rcParams.get("axes.prop_cycle")
    base_colors = None
    if color_cycle is not None:
        try:
            base_colors = color_cycle.by_key().get("color", None)
        except Exception:
            base_colors = None
    if not base_colors:
        base_colors = [plt.cm.tab10(i / max(1, len(pivot.columns) - 1)) for i in range(len(pivot.columns))]
    palette = [base_colors[i % len(base_colors)] for i, _ in enumerate(pivot.columns)]
    pivot.plot(kind="bar", ax=ax, color=palette)
    # Add thin dotted line at the highest score
    top_val = float(pivot.max().max())
    ax.axhline(top_val, linestyle=":", linewidth=0.8, color="0.3", zorder=3)
    ax.set_ylabel(metric)
    ax.set_xlabel("assay_id")
    ax.grid()
    ax.set_title(f"{metric} comparison by decomposition")
    ax.legend(title="decomposition", bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    if auto_zero:
        min_val = float(pivot.min().min())
        max_val = float(pivot.max().max())
        span = max_val - min_val
        margin = 0.05 * span if span > 0 else 0.05 * abs(min_val) if min_val != 0 else 0.1
        bottom = min_val - margin
        ymin, ymax = ax.get_ylim()
        ax.set_ylim(bottom, ymax)
    if show:
        import matplotlib.pyplot as plt
        plt.tight_layout()
        plt.show()


In [ ]:
# Use shared plotting helper
try:
    from utils import plot_embedding_2d
except ImportError:
    import sys
    from pathlib import Path
    root = Path.cwd()
    for parent in [root, *root.parents]:
        if (parent / "utils.py").exists():
            sys.path.append(str(parent))
            break
    from utils import plot_embedding_2d


In [ ]:
import time
from sklearn.metrics import roc_auc_score, average_precision_score

def perf(model, graphs, targets):
    """Return (acc, auc, ap, errors, pred_time) on graphs using model.predict[_proba]."""
    t0 = time.perf_counter()
    pred = model.predict(graphs)
    try:
        proba = model.predict_proba(graphs)
    except Exception:
        proba = None
    pred_time = time.perf_counter() - t0
    # accuracy_score imported earlier in notebook
    from sklearn.metrics import accuracy_score as _acc
    import numpy as _np
    acc = _acc(targets, pred)
    errors = int((_np.asarray(pred) != _np.asarray(targets)).sum())
    auc = ap = None
    if proba is not None:
        proba = _np.asarray(proba)
        if proba.ndim == 2 and proba.shape[1] > 1:
            if proba.shape[1] == 2:
                auc = roc_auc_score(targets, proba[:, 1])
                ap = average_precision_score(targets, proba[:, 1])
            else:
                auc = roc_auc_score(targets, proba, average='macro', multi_class='ovr')
                ap = average_precision_score(targets, proba, average='macro')
    return acc, auc, ap, errors, pred_time

def print_perf(header_str, acc, auc, ap, errors, fit_time, pred_time, n_instances):
    nps = (n_instances / (fit_time + pred_time)) if (fit_time + pred_time) > 0 else float('inf')
    print(f"{header_str}: accuracy={acc:.3f}, roc_auc={(auc if auc is not None else float('nan')):.3f}, avg_precision={(ap if ap is not None else float('nan')):.3f}, errors={errors}, fit_time={fit_time:.2f}s, pred_time={pred_time:.2f}s, total_time={(fit_time+pred_time):.2f}s, n_instances_per_second={nps:.1f}")

def plot(model, graphs, targets, graphs_te, targets_te):
    import matplotlib.pyplot as _plt
    # Full dataset
    fig, axes = _plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix='All data', mode='scatter', alpha=0.75, ax=axes[0], show=False)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix='All data', mode='knn_class_union', k=5, alpha=0.75, z=1, ax=axes[1], show=False)
    _ = plot_embedding_2d(model, graphs, targets, title_prefix='All data', mode='knn_class_union', k=11, alpha=0.75, z=1, ax=axes[2], show=False)
    _plt.show()
    # Test split
    fig, axes = _plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    _ = plot_embedding_2d(model, graphs_te, targets_te, title_prefix='Test', mode='scatter', alpha=0.75, ax=axes[0], show=False)
    _ = plot_embedding_2d(model, graphs_te, targets_te, title_prefix='Test', mode='knn_class_union', k=5, alpha=0.75, z=1, ax=axes[1], show=False)
    _ = plot_embedding_2d(model, graphs_te, targets_te, title_prefix='Test', mode='knn_class_union', k=11, alpha=0.75, z=1, ax=axes[2], show=False)
    _plt.show()


---

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")
assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
datasets = []
for assay_id in assay_ids:
            original_graphs, original_targets = loader.load(assay_id, limit_active=1500, limit_inactive=1500)
    print(f'AID{assay_id}  #graphs: {len(original_graphs)}')
    datasets.append((assay_id, original_graphs, original_targets))

---

In [ ]:
from abstractgraph.operators import *

decomposition_functions = []

cycle_and_tree = (cycle(), compose(neighborhood(radius=1), tree()))
df = add( *cycle_and_tree, compose_product(binary_combination(distance=0), *cycle_and_tree))
df = add(cycle(), tree())
decomposition_functions.append(('cycle',df))

df = neighborhood(radius=(0,3))
decomposition_functions.append(('neighborhood',df))

df = graphlet(radius=2, number_of_nodes=(3,5))
decomposition_functions.append(('graphlet',df))

def make_nsppk(radius=1, distance=3, thickness=1):
    if thickness == 0:
        df = add(union_of_shortest_paths(length=distance), compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius))))
    elif thickness == 1:
        df = compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius)))
    else:
        raise Exception(f'thickness {thickness} is not allowed')
    return df
df = make_nsppk(radius=1, distance=4, thickness=1)
decomposition_functions.append(('nsppk',df))

In [ ]:
%%time

reports = []

for assay_id, graphs, targets in datasets:
    print(f"\nProcessing assay {assay_id} (n={len(graphs)})")
    for decomposition_name, decomposition_fn in decomposition_functions:
        print(f"  Decomposition: {decomposition_name}")
        transformer = AbstractGraphTransformer(
            nbits=14,
            decomposition_function=decomposition_fn,
            return_dense=True,
            n_jobs=-1,  # avoid multiprocessing pickling issues in notebooks
        )

        graph_estimator = GraphEstimator(
            transformer=transformer,
            estimator=RandomForestClassifier(random_state=0, n_estimators=300, n_jobs=-1),
            manifold=None,
            n_selected_features=500,
        )

        # Train/test split
        start_total = time.perf_counter()
        graphs_train, graphs_test, targets_train, targets_test = train_test_split(graphs, targets, test_size=0.2, random_state=42, stratify=targets)

        start_fit = time.perf_counter()
        graph_estimator.fit(graphs_train, targets_train)
        fit_time = time.perf_counter() - start_fit
        # Evaluate on test split using helpers
        acc, auc, ap, errors, pred_time = perf(graph_estimator, graphs_test, targets_test)
        total_time = time.perf_counter() - start_total
        print_perf(f'AID {assay_id} — {decomposition_name}', acc, auc, ap, errors, fit_time, pred_time, len(graphs))

        # Visualize embeddings using helper
        plot(graph_estimator, graphs, targets, graphs_test, targets_test)


In [ ]:
report_df = pd.DataFrame(reports)
display(report_df)

plot_decomposition_comparison(report_df, metric="accuracy")
plot_decomposition_comparison(report_df, metric="roc_auc")
plot_decomposition_comparison(report_df, metric="avg_precision")

---